# Auxiliary Notebook — Composite Figures for report_new.tex

This notebook loads the individual figures produced by `main.ipynb` and combines
them into three composite figures for the condensed report (`report_new.tex`):

| Output | Contents | Replaces |
|--------|----------|----------|
| `figA_data_overview.png` | GRACE trend map + multi-signal time series | fig1, fig2 (spatial), fig3 |
| `figB_statistics.png` | CCF + Granger + rolling correlation | fig4, fig_rolling_corr |
| `figC_causal_backtest.png` | Causal chain + backtest | fig5, fig6 |

The EOF spatial loading (fig2 left panel) and spectral coherence (fig_coherence)
are described purely in text in the new report.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.gridspec as mgs
from pathlib import Path

FIG_DIR = Path("figures")
assert FIG_DIR.exists(), "Run from the repo root or adjust FIG_DIR."

# Verify all source figures are present
required = [
    "fig1_tws_trend_map.png",
    "fig3_overlay.png",
    "fig4_correlogram_granger.png",
    "fig_rolling_corr.png",
    "fig5_causal_chain.png",
    "fig6_backtest.png",
]
for f in required:
    path = FIG_DIR / f
    status = "OK" if path.exists() else "MISSING"
    print(f"  {status}  {f}")

## Figure A — Data and Regional Signal Overview

**Panel (a):** GRACE TWS linear trend map 2002–2025 with the study region bounding box.
The drying signal over the Cerrado and Paraná is visible as red areas.

**Panel (b):** All four main time series 2002–2025: GRACE TWS anomaly (blue),
MODIS cropland NDVI anomaly (green), ERA5 P-ET (purple), soybean futures (red).
Arrows indicate the ~5-month lead of TWS before the price response.

In [ ]:
img_map = mpimg.imread(FIG_DIR / "fig1_tws_trend_map.png")
img_ts  = mpimg.imread(FIG_DIR / "fig3_overlay.png")

# The map image is roughly square; the time series image is wider.
# Use width_ratios to give the time series panel more horizontal space.
fig = plt.figure(figsize=(16, 6.5))
gs  = mgs.GridSpec(1, 2, figure=fig, width_ratios=[1, 1.55], wspace=0.03)

ax_map = fig.add_subplot(gs[0])
ax_ts  = fig.add_subplot(gs[1])

ax_map.imshow(img_map, aspect="auto")
ax_map.axis("off")
ax_map.set_title(
    "(a) GRACE TWS Linear Trend 2002\u20132025\n"
    "dashed box = study region  |  red = drying trend",
    fontsize=10, pad=5,
)

ax_ts.imshow(img_ts, aspect="auto")
ax_ts.axis("off")
ax_ts.set_title(
    "(b) GRACE TWS \u00b7 MODIS NDVI \u00b7 ERA5 P-ET \u00b7 Soybean Futures 2002\u20132025\n"
    "dashed verticals = drought onset  |  arrows = ~5-month lag to market",
    fontsize=10, pad=5,
)

plt.savefig(FIG_DIR / "figA_data_overview.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figA_data_overview.png")

## Figure B — Statistical Results

**Panel (a):** Bootstrap cross-correlation (left) and Granger F-test p-values (right)
for Δ TWS → soybean log-returns at lags 0–12 months.

**Panel (b):** 60-month rolling Pearson correlation at lag 5. All windows remain
negative (mean r = −0.291), confirming that the relationship is not driven by a
single event.

In [ ]:
img_ccf  = mpimg.imread(FIG_DIR / "fig4_correlogram_granger.png")
img_roll = mpimg.imread(FIG_DIR / "fig_rolling_corr.png")

fig = plt.figure(figsize=(16, 8.5))
gs  = mgs.GridSpec(2, 1, figure=fig, height_ratios=[1, 0.78], hspace=0.10)

ax_ccf  = fig.add_subplot(gs[0])
ax_roll = fig.add_subplot(gs[1])

ax_ccf.imshow(img_ccf, aspect="auto")
ax_ccf.axis("off")
ax_ccf.set_title(
    "(a) Bootstrap Cross-Correlation (left) and Granger Causality (right): "
    "$\\Delta$TWS $\\to$ Soybean Log-Returns",
    fontsize=11, pad=5,
)

ax_roll.imshow(img_roll, aspect="auto")
ax_roll.axis("off")
ax_roll.set_title(
    "(b) 60-Month Rolling Pearson Correlation at Lag 5 Months"
    "  (mean $r = -0.291$; all windows negative throughout 2002\u20132025)",
    fontsize=11, pad=5,
)

plt.savefig(FIG_DIR / "figB_statistics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figB_statistics.png")

## Figure C — Causal Chain Analysis and Backtest

**Panel (a):** Bootstrap cross-correlations for each step of the proposed causal
chain TWS → NDVI → Soybean. Only the direct path (right sub-panel) is significant.

**Panel (b):** Illustrative long-only backtest. The strategy enters a long position
five months after TWS drops below −1σ. Sharpe 0.34 vs 0.15 for buy-and-hold;
max drawdown −24% vs −63%.

In [ ]:
img_causal   = mpimg.imread(FIG_DIR / "fig5_causal_chain.png")
img_backtest = mpimg.imread(FIG_DIR / "fig6_backtest.png")

# fig5 is tall (14×9 with a two-row GridSpec, only top half used for the CCF panels)
# Crop the bottom whitespace by trimming to the top ~55% of the image height.
h5 = img_causal.shape[0]
img_causal_cropped = img_causal[:int(h5 * 0.57), :, :]

fig = plt.figure(figsize=(16, 10))
gs  = mgs.GridSpec(2, 1, figure=fig, height_ratios=[1.1, 0.85], hspace=0.10)

ax_causal   = fig.add_subplot(gs[0])
ax_backtest = fig.add_subplot(gs[1])

ax_causal.imshow(img_causal_cropped, aspect="auto")
ax_causal.axis("off")
ax_causal.set_title(
    "(a) Bootstrap Cross-Correlation at Each Step of the Proposed Causal Chain"
    "  (gray bands = 95\u2009% CI  |  dashed verticals = best lag)",
    fontsize=11, pad=5,
)

ax_backtest.imshow(img_backtest, aspect="auto")
ax_backtest.axis("off")
ax_backtest.set_title(
    "(b) Illustrative Backtest: TWS-Signal Strategy (blue) vs Soybean Buy-and-Hold (red dashed)"
    "  |  Sharpe 0.34 vs 0.15  |  max drawdown \u221224\u2009% vs \u221263\u2009%",
    fontsize=11, pad=5,
)

plt.savefig(FIG_DIR / "figC_causal_backtest.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved figC_causal_backtest.png")

In [ ]:
print("All composite figures generated:")
for name in ["figA_data_overview.png", "figB_statistics.png", "figC_causal_backtest.png"]:
    path = FIG_DIR / name
    if path.exists():
        size_kb = path.stat().st_size / 1024
        print(f"  OK  {name}  ({size_kb:.0f} KB)")
    else:
        print(f"  MISSING  {name}")